# 02 — Model Training, Tuning & Evaluation

## Objectives

1. Build a baseline logistic-regression model as a sanity-check benchmark
2. Train an XGBoost model with Optuna Bayesian hyperparameter tuning
3. Evaluate both models with business-relevant metrics (AUC, F1, cost-sensitive metric)
4. Generate SHAP explanations for the champion model
5. Persist the trained pipeline so `app.py` can serve it

---

**Prerequisites:** Run `01_eda.ipynb` first, or make sure `data/raw/telco_churn.csv` exists  
(run `python data/download_data.py` to fetch it).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / '.env')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('Setup complete.')

## 1. Load & Prepare Data

In [ ]:
from src.pipeline.features import (
    build_preprocessor,
    clean_data,
    engineer_features,
    get_feature_target_split,
    load_data,
)

raw = load_data()
df_clean = clean_data(raw)
df_feat = engineer_features(df_clean)
X, y = get_feature_target_split(df_feat)

print(f'Feature matrix : {X.shape}')
print(f'Target          : {y.shape}  (churn rate: {y.mean():.1%})')
print(f'\nFeatures used:')
for col in X.columns:
    print(f'  {col}')

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train : {X_train.shape[0]:,} rows  (churn rate: {y_train.mean():.1%})')
print(f'Test  : {X_test.shape[0]:,} rows  (churn rate: {y_test.mean():.1%})')

## 2. Baseline — Logistic Regression

A simple, interpretable baseline tells us the minimum bar XGBoost must beat.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from sklearn.pipeline import Pipeline

baseline_pipeline = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42,
    )),
])

baseline_pipeline.fit(X_train, y_train)

y_pred_base = baseline_pipeline.predict(X_test)
y_proba_base = baseline_pipeline.predict_proba(X_test)[:, 1]

baseline_auc = roc_auc_score(y_test, y_proba_base)
baseline_f1  = f1_score(y_test, y_pred_base)

print(f'Baseline Logistic Regression')
print(f'  ROC-AUC : {baseline_auc:.4f}')
print(f'  F1      : {baseline_f1:.4f}')
print()
print(classification_report(y_test, y_pred_base, target_names=['No Churn', 'Churn']))

## 3. XGBoost + Optuna Hyperparameter Tuning

We use `src.pipeline.train.train_model` which wraps the full Optuna + MLflow pipeline.
Set `n_trials=5` for a quick demo; use 20+ for production quality.

In [ ]:
from src.pipeline.train import train_model

# Pass the cleaned + engineered df directly to skip re-downloading
champion_pipeline, metrics = train_model(df=df_clean, n_trials=20)

print(f'\nChampion XGBoost')
print(f'  ROC-AUC   : {metrics["auc"]:.4f}   (baseline: {baseline_auc:.4f})')
print(f'  F1        : {metrics["f1"]:.4f}   (baseline: {baseline_f1:.4f})')
print(f'  Precision : {metrics["precision"]:.4f}')
print(f'  Recall    : {metrics["recall"]:.4f}')
print(f'  CV AUC    : {metrics.get("cv_auc", "n/a")}')

## 4. Model Comparison — ROC Curves

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(8, 6))

RocCurveDisplay.from_estimator(
    baseline_pipeline, X_test, y_test,
    name=f'Logistic Regression (AUC={baseline_auc:.3f})',
    color='#3498db', ax=ax,
)
RocCurveDisplay.from_estimator(
    champion_pipeline, X_test, y_test,
    name=f'XGBoost + Optuna (AUC={metrics["auc"]:.3f})',
    color='#e74c3c', ax=ax,
)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random (AUC=0.500)')
ax.set_title('ROC Curves — Baseline vs Champion', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

## 5. Confusion Matrix & Classification Report

In [ ]:
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

y_pred_champ = champion_pipeline.predict(X_test)
cm = confusion_matrix(y_test, y_pred_champ)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Counts
disp1 = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['No Churn', 'Churn'],
)
disp1.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix — Counts', fontsize=12, fontweight='bold')

# Normalised
cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
disp2 = ConfusionMatrixDisplay(
    confusion_matrix=cm_norm,
    display_labels=['No Churn', 'Churn'],
)
disp2.plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Confusion Matrix — Normalised', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_champ, target_names=['No Churn', 'Churn']))

## 6. Business Cost Metric

Standard metrics treat all errors equally. In churn prediction:
- **False Negative** (missing a churner) = lost customer = ~\$100 revenue lost
- **False Positive** (flagging a loyal customer) = wasted retention offer = ~\$10

We compute the total cost under these assumptions and compare models.

In [ ]:
COST_FP = 10   # USD: retention offer sent to loyal customer
COST_FN = 100  # USD: missed churner — lost MRR

def business_cost(y_true, y_pred, cost_fp=COST_FP, cost_fn=COST_FN):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    total_cost = (fp * cost_fp) + (fn * cost_fn)
    return total_cost, fp, fn, tp, tn

cost_base, fp_b, fn_b, tp_b, tn_b = business_cost(y_test, y_pred_base)
cost_champ, fp_c, fn_c, tp_c, tn_c = business_cost(y_test, y_pred_champ)

print('=== Business Cost Analysis (test set) ===')
print(f'\n  Logistic Regression baseline:')
print(f'    False Positives : {fp_b:3d}  (cost: ${fp_b * COST_FP:,})')
print(f'    False Negatives : {fn_b:3d}  (cost: ${fn_b * COST_FN:,})')
print(f'    Total cost      : ${cost_base:,}')

print(f'\n  XGBoost champion:')
print(f'    False Positives : {fp_c:3d}  (cost: ${fp_c * COST_FP:,})')
print(f'    False Negatives : {fn_c:3d}  (cost: ${fn_c * COST_FN:,})')
print(f'    Total cost      : ${cost_champ:,}')

saving = cost_base - cost_champ
print(f'\n  Cost saving vs baseline : ${saving:,} ({saving/cost_base*100:.1f}% reduction)')

## 7. Optimal Decision Threshold

The default threshold of 0.5 is arbitrary. We sweep thresholds to minimise business cost.

In [ ]:
y_proba_champ = champion_pipeline.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.1, 0.9, 0.02)
costs, f1s, precisions, recalls = [], [], [], []

for t in thresholds:
    y_pred_t = (y_proba_champ >= t).astype(int)
    cost, *_ = business_cost(y_test, y_pred_t)
    costs.append(cost)
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))
    from sklearn.metrics import precision_score, recall_score
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))

optimal_idx = np.argmin(costs)
optimal_threshold = thresholds[optimal_idx]
optimal_cost = costs[optimal_idx]

print(f'Optimal threshold for minimum business cost: {optimal_threshold:.2f}')
print(f'Cost at optimal threshold : ${optimal_cost:,}')
print(f'Cost at default (0.50)    : ${costs[np.searchsorted(thresholds, 0.5)]:,}')

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

ax1.plot(thresholds, costs, color='#e74c3c', linewidth=2.5, label='Business cost ($)')
ax1.axvline(optimal_threshold, color='#e74c3c', linestyle='--', alpha=0.6,
            label=f'Optimal threshold ({optimal_threshold:.2f})')
ax1.set_xlabel('Decision Threshold', fontsize=12)
ax1.set_ylabel('Total Business Cost ($)', fontsize=12, color='#e74c3c')
ax1.tick_params(axis='y', labelcolor='#e74c3c')

ax2.plot(thresholds, f1s, color='#2ecc71', linewidth=2, label='F1-score')
ax2.plot(thresholds, recalls, color='#3498db', linewidth=1.5, linestyle=':', label='Recall')
ax2.set_ylabel('Score', fontsize=12)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)

ax1.set_title('Threshold Sweep — Business Cost vs F1', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. SHAP Feature Importance

SHAP (SHapley Additive exPlanations) gives us a theoretically grounded decomposition of each prediction. Unlike simple feature importance from the tree splits, SHAP values tell us both *which* features matter and *in which direction*.

In [ ]:
import shap

clf = champion_pipeline.named_steps['classifier']
pre = champion_pipeline.named_steps['preprocessor']

X_test_transformed = pre.transform(X_test)

try:
    feature_names = list(pre.get_feature_names_out())
except Exception:
    feature_names = [f'f{i}' for i in range(X_test_transformed.shape[1])]

explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test_transformed)

# For binary classification, shap_values may be a list [class0, class1]
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

print(f'SHAP values computed for {sv.shape[0]} test samples × {sv.shape[1]} features')

In [ ]:
mean_abs_shap = np.abs(sv).mean(axis=0)
top_n = 20
top_idx = np.argsort(mean_abs_shap)[-top_n:][::-1]

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#e74c3c' if mean_abs_shap[i] > mean_abs_shap[top_idx[top_n // 2]] else '#3498db'
          for i in top_idx]

ax.barh(
    [feature_names[i] if i < len(feature_names) else f'f{i}' for i in top_idx],
    mean_abs_shap[top_idx],
    color=colors,
    edgecolor='none',
)
ax.set_xlabel('Mean |SHAP value|', fontsize=12)
ax.set_title(f'Top {top_n} Feature Importances (SHAP)', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# SHAP beeswarm — shows direction of each feature's effect
import shap
import pandas as pd

X_test_df = pd.DataFrame(X_test_transformed, columns=feature_names)
explanation = shap.Explanation(
    values=sv,
    base_values=explainer.expected_value if not isinstance(explainer.expected_value, list)
               else explainer.expected_value[1],
    data=X_test_df.values,
    feature_names=feature_names,
)

shap.plots.beeswarm(explanation, max_display=15, show=False)
plt.title('SHAP Beeswarm — Top 15 Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Precision–Recall Curve

In imbalanced datasets, the PR curve is more informative than the ROC curve because it focuses on the minority class (churners).

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, average_precision_score

ap_score = average_precision_score(y_test, y_proba_champ)

fig, ax = plt.subplots(figsize=(7, 6))
PrecisionRecallDisplay.from_predictions(
    y_test, y_proba_champ,
    name=f'XGBoost (AP={ap_score:.3f})',
    color='#e74c3c', ax=ax,
)

# Baseline: random classifier precision = prevalence
prevalence = y_test.mean()
ax.axhline(prevalence, color='gray', linestyle='--', alpha=0.6,
           label=f'Random (AP={prevalence:.3f})')

ax.set_title('Precision–Recall Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 10. Save the Champion Model

The full sklearn Pipeline (preprocessor + XGBClassifier) is serialised with `joblib`.  
`app.py` loads it from `models/best_model.joblib` at startup.

In [ ]:
from src.config import BEST_MODEL_PATH

BEST_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(champion_pipeline, BEST_MODEL_PATH)

print(f'Model saved to: {BEST_MODEL_PATH}')
print(f'File size: {BEST_MODEL_PATH.stat().st_size / 1024:.1f} KB')

# Quick round-trip verification
loaded = joblib.load(BEST_MODEL_PATH)
loaded_proba = loaded.predict_proba(X_test)[:, 1]
assert abs(roc_auc_score(y_test, loaded_proba) - metrics['auc']) < 1e-6, 'AUC mismatch after reload!'
print('Reload verification passed.')

## 11. Summary

### Model Performance

| Model | ROC-AUC | F1 | Business Cost |
|---|---|---|---|
| Logistic Regression (baseline) | ~0.84 | ~0.59 | higher |
| **XGBoost + Optuna (champion)** | **~0.85** | **~0.62** | **lower** |

### Top Churn Predictors (SHAP)

1. **Contract type** — Month-to-month is by far the strongest churn signal
2. **Tenure** — Shorter tenure = higher churn risk
3. **MonthlyCharges / AvgMonthlySpend** — Higher bills correlate with churn
4. **InternetService (Fiber optic)** — Fiber users churn twice as often
5. **OnlineSecurity / TechSupport** — Absence of these add-ons is a risk flag

### Next Steps

- Launch the Gradio demo: `python app.py`
- Use the LangGraph pipeline for automated batch analysis: see `src/graph/churn_graph.py`
- Ask natural-language questions in the Analyze tab (requires `GROQ_API_KEY`)